In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
columns = [
    "id",
    "label",
    "statement",
    "subject",
    "speaker",
    "speaker_job_title",
    "state_info",
    "party_affiliation",
    "barely_true_counts",
    "false_counts",
    "half_true_counts",
    "mostly_true_counts",
    "pants_on_fire_counts",
    "context"
]

In [4]:
train_df = pd.read_csv(r"C:\Users\abenech\Downloads\train.tsv", sep="\t", header=None, names=columns)
valid_df = pd.read_csv(r"C:\Users\abenech\Downloads\valid.tsv", sep="\t", header=None, names=columns)
test_df = pd.read_csv(r"C:\Users\abenech\Downloads\test.tsv", sep="\t", header=None, names=columns)

In [5]:
print("Train shape :", train_df.shape)
print("Valid shape :", valid_df.shape)
print("Test shape  :", test_df.shape)

train_df.head()

Train shape : (10240, 14)
Valid shape : (1284, 14)
Test shape  : (1267, 14)


,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN


In [6]:
print("Répartition des labels dans train :")
print(train_df["label"].value_counts())

Répartition des labels dans train :
label
half-true      2114
false          1995
mostly-true    1962
true           1676
barely-true    1654
pants-fire      839
Name: count, dtype: int64


In [7]:
print("Valeurs manquantes dans train :")
print(train_df.isna().sum())

print("\nValeurs manquantes dans valid :")
print(valid_df.isna().sum())

print("\nValeurs manquantes dans test :")
print(test_df.isna().sum())

Valeurs manquantes dans train :
id                         0
label                      0
statement                  0
subject                    2
speaker                    2
speaker_job_title       2898
state_info              2210
party_affiliation          2
barely_true_counts         2
false_counts               2
half_true_counts           2
mostly_true_counts         2
pants_on_fire_counts       2
context                  102
dtype: int64

Valeurs manquantes dans valid :
id                        0
label                     0
statement                 0
subject                   0
speaker                   0
speaker_job_title       345
state_info              279
party_affiliation         0
barely_true_counts        0
false_counts              0
half_true_counts          0
mostly_true_counts        0
pants_on_fire_counts      0
context                  12
dtype: int64

Valeurs manquantes dans test :
id                        0
label                     0
statement              

In [8]:
#Pas de valeurs importantes manquantes on ne supprime pas de ligne

In [9]:
#On transforme les labels en binaire en simplifiant le pb
label_mapping = {
    "pants-fire": "fake",
    "false": "fake",
    "barely-true": "fake",
    "mostly-true": "real",
    "true": "real"
}

In [10]:
#Retirer les lignes half-true puis mapping
train_df = train_df[train_df["label"] != "half-true"].copy()
valid_df = valid_df[valid_df["label"] != "half-true"].copy()
test_df = test_df[test_df["label"] != "half-true"].copy()

train_df["label_binary"] = train_df["label"].map(label_mapping)
valid_df["label_binary"] = valid_df["label"].map(label_mapping)
test_df["label_binary"] = test_df["label"].map(label_mapping)

In [11]:
#Vérifie les résultats du mapping
print("Train :")
print(train_df["label_binary"].value_counts())

print("\nValid :")
print(valid_df["label_binary"].value_counts())

print("\nTest :")
print(test_df["label_binary"].value_counts())

Train :
label_binary
fake    4488
real    3638
Name: count, dtype: int64

Valid :
label_binary
fake    616
real    420
Name: count, dtype: int64

Test :
label_binary
fake    553
real    449
Name: count, dtype: int64


In [12]:
X_train = train_df["statement"]
y_train = train_df["label_binary"]

X_valid = valid_df["statement"]
y_valid = valid_df["label_binary"]

X_test = test_df["statement"]
y_test = test_df["label_binary"]

In [13]:
#Transformer le texte avec TF-IDF
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_valid_tfidf = vectorizer.transform(X_valid)
X_test_tfidf = vectorizer.transform(X_test)

In [14]:
#Entraîner un premier modèle
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [15]:
#Faire des prédictions sur validation
y_valid_pred = model.predict(X_valid_tfidf)

print("Accuracy sur validation :", accuracy_score(y_valid, y_valid_pred))
print("\nClassification report sur validation :\n")
print(classification_report(y_valid, y_valid_pred))

Accuracy sur validation : 0.6505791505791506

Classification report sur validation :

              precision    recall  f1-score   support

        fake       0.69      0.76      0.72       616
        real       0.58      0.49      0.53       420

    accuracy                           0.65      1036
   macro avg       0.63      0.63      0.63      1036
weighted avg       0.64      0.65      0.64      1036



In [16]:
#Faire des prédictions sur test
y_test_pred = model.predict(X_test_tfidf)

print("Accuracy sur test :", accuracy_score(y_test, y_test_pred))
print("\nClassification report sur test :\n")
print(classification_report(y_test, y_test_pred))

Accuracy sur test : 0.626746506986028

Classification report sur test :

              precision    recall  f1-score   support

        fake       0.64      0.73      0.68       553
        real       0.60      0.51      0.55       449

    accuracy                           0.63      1002
   macro avg       0.62      0.62      0.62      1002
weighted avg       0.62      0.63      0.62      1002



In [19]:
#Matrice de confusion
cm = confusion_matrix(y_test, y_test_pred, labels=["fake", "real"])
cm_df = pd.DataFrame(cm, index=["fake", "real"], columns=["fake prédit", "real prédit"])
cm_df

,fake prédit,real prédit
fake,401,152
real,222,227


In [20]:
#Exemples
results_df = pd.DataFrame({
    "statement": X_test.values,
    "true_label": y_test.values,
    "predicted_label": y_test_pred
})

results_df.head(10)

,statement,true_label,predicted_label
0,Building a wall on the U.S.-Mexico border will...,real,fake
1,Wisconsin is on pace to double the number of l...,fake,fake
2,Says John McCain has done nothing to help the ...,fake,fake
3,When asked by a reporter whether hes at the ce...,fake,fake
4,Over the past five years the federal governmen...,real,real
5,Says that Tennessee law requires that schools ...,real,real
6,"Says Vice President Joe Biden ""admits that the...",fake,fake
7,Donald Trump is against marriage equality. He ...,real,real
8,We know that more than half of Hillary Clinton...,fake,fake
9,We know there are more Democrats in Georgia th...,fake,fake
